## Logistic Regression for Feature Importance,Segment Analysis and ranking 

In [87]:
import pandas as pd


In [88]:
df = pd.read_csv("../Data/Cleaned/customer_churn_dataset.csv")
df.head(5)

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1.0,22.0,Female,25.0,14.0,4.0,27.0,Basic,Monthly,598.0,9.0,1.0
1,2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0
2,2.0,41.0,Female,28.0,28.0,7.0,13.0,Standard,Monthly,584.0,20.0,0.0
3,3.0,47.0,Male,27.0,10.0,2.0,29.0,Premium,Annual,757.0,21.0,0.0
4,3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0


In [89]:
import numpy as np

df["Issue_Level"] = np.where(
    df["Support Calls"] <= 2,
    "Low Issues",
    np.where(df["Support Calls"] <= 4,
             "Medium Issues",
             "High Issues")
)

df["Delay_Level"] = np.where(
    df["Payment Delay"] <= 15,
    "Low Delay",
    np.where(df["Payment Delay"] <= 20,
             "Medium Delay",
             "High Delay")
)

df["Spend_Level"] = np.where(
    df["Total Spend"] <= 508,
    "Low Spend",
    "High Spend"
)

In [90]:
df.duplicated().sum()

np.int64(0)

In [91]:
len(df)

505206

In [92]:
df["CustomerID"].nunique()

442211

In [93]:
df["CustomerID"].duplicated().sum()

np.int64(62995)

In [94]:
df = df.drop(columns=["CustomerID"])

In [95]:
df.head()

,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn,Issue_Level,Delay_Level,Spend_Level
0,22.0,Female,25.0,14.0,4.0,27.0,Basic,Monthly,598.0,9.0,1.0,Medium Issues,High Delay,High Spend
1,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.0,17.0,1.0,High Issues,Medium Delay,High Spend
2,41.0,Female,28.0,28.0,7.0,13.0,Standard,Monthly,584.0,20.0,0.0,High Issues,Low Delay,High Spend
3,47.0,Male,27.0,10.0,2.0,29.0,Premium,Annual,757.0,21.0,0.0,Low Issues,High Delay,High Spend
4,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.0,6.0,1.0,High Issues,Low Delay,High Spend


##### Observation and Conclusion

1. There are multiple duplicates of customerID's but the attributes are different which suggests dataset creater might have accidently reused them that's why I dropped it.

## Important Features using Logistic Regression 
- I used odds ratios to identify the contribution of feature segments to customer churn.



In [96]:
df.columns.tolist()

['Age',
 'Gender',
 'Tenure',
 'Usage Frequency',
 'Support Calls',
 'Payment Delay',
 'Subscription Type',
 'Contract Length',
 'Total Spend',
 'Last Interaction',
 'Churn',
 'Issue_Level',
 'Delay_Level',
 'Spend_Level']

In [97]:
# Features
X = df[
    [
        "Issue_Level",
        "Delay_Level",
        "Spend_Level",
        "Contract Length"
    ]
]

# Target
y = df["Churn"]

# One-Hot Encoding
X_encoded = pd.get_dummies(
    X,
    drop_first=True,
    dtype=int
)

X_encoded.head()

,Issue_Level_Low Issues,Issue_Level_Medium Issues,Delay_Level_Low Delay,Delay_Level_Medium Delay,Spend_Level_Low Spend,Contract Length_Monthly,Contract Length_Quarterly
0,0,1,0,0,0,1,0
1,0,0,0,1,0,0,0
2,0,0,1,0,0,1,0
3,1,0,0,0,0,0,0
4,0,0,1,0,0,1,0


In [98]:
X_encoded.columns.tolist()

['Issue_Level_Low Issues',
 'Issue_Level_Medium Issues',
 'Delay_Level_Low Delay',
 'Delay_Level_Medium Delay',
 'Spend_Level_Low Spend',
 'Contract Length_Monthly',
 'Contract Length_Quarterly']

In [99]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [100]:
from sklearn.linear_model import LogisticRegression

model1 = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model1.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [101]:
y_pred = model1.predict(X_test)

In [102]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

Accuracy : 0.8851566675243958
Precision: 0.887634380499364
Recall   : 0.9081088789461488
F1 Score : 0.8977549078349135


In [105]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_encoded, y)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [106]:
coef_df = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Coefficient": model.coef_[0]
})

coef_df.sort_values(
    by="Coefficient",
    ascending=False
)

,Feature,Coefficient
4,Spend_Level_Low Spend,2.043516
5,Contract Length_Monthly,1.988643
6,Contract Length_Quarterly,-0.012846
1,Issue_Level_Medium Issues,-2.228073
3,Delay_Level_Medium Delay,-2.540315
2,Delay_Level_Low Delay,-2.765522
0,Issue_Level_Low Issues,-2.847519


In [107]:
import numpy as np

odds_df = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Coefficient": model.coef_[0],
    "Odds_Ratio": np.exp(model.coef_[0])
})

odds_df.sort_values(
    by="Odds_Ratio",
    ascending=False
)

,Feature,Coefficient,Odds_Ratio
4,Spend_Level_Low Spend,2.043516,7.717694
5,Contract Length_Monthly,1.988643,7.305611
6,Contract Length_Quarterly,-0.012846,0.987236
1,Issue_Level_Medium Issues,-2.228073,0.107736
3,Delay_Level_Medium Delay,-2.540315,0.078842
2,Delay_Level_Low Delay,-2.765522,0.062943
0,Issue_Level_Low Issues,-2.847519,0.057988


#### Observations 

1. The low spend customers churn 7.7 times higher than high spend customers.
2. The Customers with the monthly contract length churn 7.3 times higher than customer with annual contract length.
    - Where annual and qurterly customers churn almost equivalently.
3. The High issues customers churn 17.54 times higher than customer with low issues.
4. The customers with High delay churn almost 16 times higher than low delay customers.

#### Conclusion
- So, the high churn segments are
    1. Montly contract
    2. Low spend
    3. High issues 
    4. High delay


## Segment Analysis

In [108]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X = pd.get_dummies(df[["Contract Length"]], drop_first=True)
y = (df["Spend_Level"] == "Low Spend").astype(int)

model_contract_spend = LogisticRegression(max_iter=1000)
model_contract_spend.fit(X, y)

pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model_contract_spend.coef_[0],
    "Odds_Ratio": np.exp(model_contract_spend.coef_[0])
}).sort_values("Odds_Ratio", ascending=False)

,Feature,Coefficient,Odds_Ratio
0,Contract Length_Monthly,0.859604,2.362225
1,Contract Length_Quarterly,-0.008225,0.991809


#### Observations
- The monthly Contract length customers are 2.36 times more low spend customers than annual/quarterly customers. 

In [109]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X = pd.get_dummies(df[["Contract Length"]], drop_first=True)
y = (df["Spend_Level"] == "High Spend").astype(int)

model_contract_spend = LogisticRegression(max_iter=1000)
model_contract_spend.fit(X, y)

pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model_contract_spend.coef_[0],
    "Odds_Ratio": np.exp(model_contract_spend.coef_[0])
}).sort_values("Odds_Ratio", ascending=False)

,Feature,Coefficient,Odds_Ratio
1,Contract Length_Quarterly,0.008225,1.008259
0,Contract Length_Monthly,-0.859604,0.423330


#### Observations
- The annual/quarterly Contract length customers are 2.38 times more high spend customers than monthly customers. 

In [110]:
X = pd.get_dummies(
    df[["Spend_Level", "Contract Length","Delay_Level"]],
    drop_first=True
)

y = (df["Issue_Level"] == "High Issues").astype(int)

model_issue = LogisticRegression(max_iter=1000)
model_issue.fit(X, y)

pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model_issue.coef_[0],
    "Odds_Ratio": np.exp(model_issue.coef_[0])
}).sort_values("Odds_Ratio", ascending=False)

,Feature,Coefficient,Odds_Ratio
0,Spend_Level_Low Spend,0.903048,2.467111
1,Contract Length_Monthly,0.777611,2.176268
2,Contract Length_Quarterly,-0.007230,0.992796
4,Delay_Level_Medium Delay,-0.764005,0.465797
3,Delay_Level_Low Delay,-0.891879,0.409885


#### Observations

1. The low Spend Customers face 2.46 times higher issues than high spend customers
2. The montly contract length customers face 2.17 time higher issues than annual/quarterly customers
3. The high delay customers face 2.5 times higher than low delay customers


## Rankng of Importance of Feature
- I used the the Cramér's V for the association strength of features with churn to rank their importance 
-  Ranking can help decide which features should priortised while preventing the churn


In [137]:
import pandas as pd
import scipy.stats as stats
import numpy as np

def cramers_v(confusion_matrix):
    chi2 = stats.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, c = confusion_matrix.shape
    return np.sqrt(chi2 / (n * (min(r, c) - 1)))

def validate_relationship(df, feature1, feature2):
    
    table = pd.crosstab(df[feature1], df[feature2])

    chi2, p, dof, expected = stats.chi2_contingency(table)

    cv = cramers_v(table)

    print("="*60)
    print(f"{feature1}  →  {feature2}")
    print("="*60)
    print(table)
    print()
    print(f"Chi-Square Statistic : {chi2:.3f}")
    print(f"P-value              : {p:.6f}")
    print(f"Cramer's V           : {cv:.3f}")
    print()

In [138]:
validate_relationship(
    df,
    "Contract Length",
    "Spend_Level"
)

Contract Length  →  Spend_Level
Spend_Level      High Spend  Low Spend
Contract Length                       
Annual               147155      51453
Monthly               59856      49378
Quarterly            146504      50860

Chi-Square Statistic : 15282.791
P-value              : 0.000000
Cramer's V           : 0.174



In [139]:
validate_relationship(
    df,
    "Spend_Level",
    "Issue_Level"
)

Spend_Level  →  Issue_Level
Issue_Level  High Issues  Low Issues  Medium Issues
Spend_Level                                        
High Spend        100321      178819          74375
Low Spend          82905       41811          26975

Chi-Square Statistic : 33647.124
P-value              : 0.000000
Cramer's V           : 0.258



In [140]:
validate_relationship(
    df,
    "Contract Length",
    "Issue_Level"
)

Contract Length  →  Issue_Level
Issue_Level      High Issues  Low Issues  Medium Issues
Contract Length                                        
Annual                 61652       95887          41069
Monthly                60637       29208          19389
Quarterly              60937       95535          40892

Chi-Square Statistic : 23752.102
P-value              : 0.000000
Cramer's V           : 0.153



In [141]:
validate_relationship(
    df,
    "Issue_Level",
    "Delay_Level"
)

Issue_Level  →  Delay_Level
Delay_Level    High Delay  Low Delay  Medium Delay
Issue_Level                                       
High Issues         63500      88369         31357
Low Issues          28925     146105         45600
Medium Issues       19284      62514         19552

Chi-Square Statistic : 27919.224
P-value              : 0.000000
Cramer's V           : 0.166



In [142]:
validate_relationship(
    df,
    "Delay_Level",
    "Churn"
)

Delay_Level  →  Churn
Churn            0.0     1.0
Delay_Level                 
High Delay      6478  105231
Low Delay     167960  129028
Medium Delay   50276   46233

Chi-Square Statistic : 87480.685
P-value              : 0.000000
Cramer's V           : 0.416



In [143]:
validate_relationship(
    df,
    "Spend_Level",
    "Churn"
)

Spend_Level  →  Churn
Churn           0.0     1.0
Spend_Level                
High Spend   207011  146504
Low Spend     17703  133988

Chi-Square Statistic : 94491.019
P-value              : 0.000000
Cramer's V           : 0.432



In [144]:
validate_relationship(
    df,
    "Issue_Level",
    "Churn"
)

Issue_Level  →  Churn
Churn             0.0     1.0
Issue_Level                  
High Issues     16876  166350
Low Issues     153926   66704
Medium Issues   53912   47438

Chi-Square Statistic : 152535.693
P-value              : 0.000000
Cramer's V           : 0.549



In [145]:
validate_relationship(
    df,
    "Contract Length",
    "Churn"
)

Contract Length  →  Churn
Churn               0.0    1.0
Contract Length               
Annual           107067  91541
Monthly           10709  98525
Quarterly        106938  90426

Chi-Square Statistic : 67861.647
P-value              : 0.000000
Cramer's V           : 0.367



#### Observation and conclusion
-  The ranking is as follows
 1. Issue level with Cramér's V of 0.549
 2. Spend level with Cramér's V of 0.432
 3. Delay level with Cramér's V of 0.416
 4. Contract Length with Cramér's V of 0.367

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

# -------------------------------
# Function to train logistic model
# -------------------------------

def get_odds_ratios(features, model_name):

    X = pd.get_dummies(
        df[features],
        drop_first=True,
        dtype=int
    )

    y = df["Churn"]

    model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    model.fit(X, y)

    odds = pd.DataFrame({
        "Feature": X.columns,
        model_name: np.exp(model.coef_[0])
    })

    return odds


# -------------------------------
# Train all models
# -------------------------------

m1 = get_odds_ratios(
    ["Contract Length"],
    "Model 1\n(Contract)"
)

m2 = get_odds_ratios(
    ["Contract Length", "Delay_Level"],
    "Model 2"
)

m3 = get_odds_ratios(
    ["Contract Length", "Spend_Level", "Delay_Level"],
    "Model 3"
)

m4 = get_odds_ratios(
    ["Contract Length", "Spend_Level", "Issue_Level", "Delay_Level"],
    "Model"
)

# -------------------------------
# Merge all tables
# -------------------------------

comparison = (
    m1
    .merge(m2, on="Feature", how="outer")
    .merge(m3, on="Feature", how="outer")
    .merge(m4, on="Feature", how="outer")
)

comparison = comparison.fillna("-")

comparison

,Feature,Model 1\n(Contract),Model 2,Model 3,Model
0,Contract Length_Monthly,10.756626,10.551311,9.67373,7.305611
1,Contract Length_Quarterly,0.988779,0.986251,0.987018,0.987236
2,Delay_Level_Low Delay,-,0.047915,0.050868,0.062943
3,Delay_Level_Medium Delay,-,0.058627,0.064133,0.078842
4,Issue_Level_Low Issues,-,-,-,0.057988
5,Issue_Level_Medium Issues,-,-,-,0.107736
6,Spend_Level_Low Spend,-,-,9.845029,7.717694
